# Team agent benchmark · one Golden set, interchangeable AgentWrappers

Every participating team provides an importable Python wrapper class with
`start()`, `answer(question)` and `close()`. The notebook knows **nothing**
about any agent SDK or implementation and treats the cli-agent wrapper
**exactly like the other agent wrappers**.

An example implementation, `cli_agent_wrapper.py`, lives **beside this
notebook** and is loaded through the same generic factory as every team.
The loader supports both adjacent single-name modules and installed
dotted Python packages. All cli-agent SDK, Knowledge, tool telemetry and
configuration logic lives in that file.

The benchmark initializes each wrapper once per run (or after a failure),
asks the same immutable Golden questions in independent conversations,
caches the answers and lets DeepEval judge them afterward.

## 1 · Configure agents and judge

The only requirement for each agent is an importable wrapper class with an
async lifecycle. Its constructor receives optional `kwargs`. Every Golden
starts in an independent conversation, even when a wrapper reuses its
initialized resources.

The supplied `cli-agent` entry uses the same `module:Class` import logic
as `team-b` and other agents. Its SDK workspace defaults to `"."`,
independent of where the Goldens are stored, and can be changed using
`kwargs["workspace"]`. There is no SDK-specific branch in this notebook.

`revision` must change when the agent implementation, model, prompt or
external sources change. To also hash arbitrary local configuration files,
use the **generic** `cache_files` option for any agent.
`MAX_CASES=3` is a smoke test; set it to `None` for all Goldens.

In [ ]:
from pathlib import Path
import asyncio
import hashlib
import importlib
import importlib.util
import inspect
import json
import os
import re
import sys
import tempfile
from collections.abc import Mapping
from typing import Protocol

# Jupyter normally sets cwd to the notebook directory; a repository-root
# launch works as well. This is strictly a BENCHMARK data path, unrelated to
# the workspace of any participating agent.
KERNEL_DIR = Path.cwd().resolve()
NOTEBOOK_DIR = next(
    (directory for directory in (
        KERNEL_DIR, KERNEL_DIR / "flow" / "rag" / "generate_goldens",
    ) if (directory / "benchmark.ipynb").is_file()),
    KERNEL_DIR,
)
GOLDENS_FILE = (NOTEBOOK_DIR / "../../../rag-goldens/deepeval-goldens.json").resolve()
OUTPUT_DIR = GOLDENS_FILE.parent / "evaluation"
# Override GOLDENS_FILE if the Goldens were exported to a different location.

AGENTS = {
    "cli-agent": {
        "entrypoint": "cli_agent_wrapper:CliAgentWrapper",
        "revision": "sdk-baseline-v1",
        "kwargs": {
            "workspace": ".",  # SDK workspace; independent of Golden data
            "include_tool_arguments": True,  # explicit opt-in for SDK tracing
            # Replace these placeholders with your actual Knowledge sources:
            # "okf": {"repository": "/REPLACE_WITH_OKF_REPOSITORY"},
            # "rag": {
            #     "mode": "okf", "database": "REPLACE_WITH_VECTOR_DATABASE",
            #     "provider": "ollama", "model": "REPLACE_WITH_EMBEDDING_MODEL",
            #     "base_url": "http://REPLACE_WITH_EMBEDDING_HOST.example:11434",
            # },
            # To disable both Knowledge integrations: "okf": None, "rag": None
        },
        # Optional generic config/prompt files to include in cache hash:
        # "cache_files": ["my-config.toml", ".cli-agent/system-prompt.md"],
        "style_criteria": None,
    },
    # "team-b": {
    #     "entrypoint": "team_b.benchmark:TeamBWrapper",
    #     "revision": "commit-a1b2c3d",
    #     "kwargs": {"endpoint": "http://localhost:8080"},
    #     "style_criteria": None,
    # },
}

MAX_CASES = 3
TIMEOUT_SECONDS = 600
FRESH_RESPONSES = False
CACHE_EPOCH = ""  # update after changing corpus, indexes or external services

# Same independently chosen judge for every agent
JUDGE_PROVIDER = "openrouter"  # openrouter | openai | ollama
JUDGE_MODEL = "openai/gpt-4.1"
JUDGE_BASE_URL = None
THRESHOLD = 0.7
JUDGE_MAX_CONCURRENT = 4
FRESH_EVALUATION = False
# Tool matching is optional: Goldens need expected_tools to enable it.
# With False, ToolCorrectness compares tool identities, not input arguments.
TOOL_COMPARE_ARGUMENTS = False

## 2 · Load one immutable DeepEval dataset and an independent judge

No agent code is needed to load the dataset. DeepEval owns only the
**evaluation stage**: the agents never see `expected_output`.

In [ ]:
from deepeval.dataset import EvaluationDataset
from deepeval.models import OpenAIModel, OpenRouterModel, OllamaModel

dataset = EvaluationDataset()
dataset.add_goldens_from_json_file(str(GOLDENS_FILE))
if MAX_CASES is not None:
    dataset = EvaluationDataset(goldens=dataset.goldens[:MAX_CASES])
if not dataset.goldens:
    raise ValueError("The shared Golden set is empty")
if len({g.additional_metadata["golden_id"] for g in dataset.goldens}) != len(dataset.goldens):
    raise ValueError("Duplicate golden_id in the dataset")

if not AGENTS or any(not re.fullmatch(r"[a-z][a-z0-9_-]{0,40}", name) for name in AGENTS):
    raise ValueError("Agent IDs must be unique lowercase identifiers")
for name, spec in AGENTS.items():
    if not isinstance(spec, dict):
        raise TypeError(f"{name}: agent specification must be a dictionary")
    if not isinstance(spec.get("entrypoint"), str) or not spec["entrypoint"].strip():
        raise ValueError(f"{name}: entrypoint is required")
    if not isinstance(spec.get("revision"), str) or not spec["revision"].strip():
        raise ValueError(f"{name}: nonempty revision is required for cache correctness")
    style = spec.get("style_criteria")
    if style is not None and (not isinstance(style, str) or not style.strip()):
        raise ValueError(f"{name}: style_criteria must be text or None")
    kwargs = spec.get("kwargs", {})
    if not isinstance(kwargs, dict) or not all(
        isinstance(k, str) for k in kwargs
    ):
        raise ValueError(f"{name}: kwargs must be a dictionary with string keys")
    try:
        json.dumps(kwargs, sort_keys=True, allow_nan=False)
    except (TypeError, ValueError) as exc:
        raise ValueError(f"{name}: kwargs must be JSON-serializable for caching") from exc
    cache_files = spec.get("cache_files", [])
    if (not isinstance(cache_files, (list, tuple)) or len(cache_files) > 32
            or any(not isinstance(p, str) or not p.strip() for p in cache_files)):
        raise ValueError(f"{name}: cache_files must list up to 32 file paths")

if JUDGE_PROVIDER == "openrouter":
    if JUDGE_BASE_URL:
        raise ValueError("OpenRouter uses its fixed endpoint")
    judge = OpenRouterModel(model=JUDGE_MODEL, api_key=os.environ["OPENROUTER_API_KEY"])
elif JUDGE_PROVIDER == "openai":
    judge = OpenAIModel(
        model=JUDGE_MODEL, api_key=os.environ["OPENAI_API_KEY"],
        **({"base_url": JUDGE_BASE_URL} if JUDGE_BASE_URL else {}),
    )
elif JUDGE_PROVIDER == "ollama":
    judge = OllamaModel(
        model=JUDGE_MODEL,
        **({"base_url": JUDGE_BASE_URL} if JUDGE_BASE_URL else {}),
    )
else:
    raise ValueError(f"Unsupported judge provider: {JUDGE_PROVIDER}")
print(f"{len(dataset.goldens)} identical Goldens | {len(AGENTS)} agents | judge={JUDGE_MODEL}")

## 3 · AgentWrapper interface and factory

All participants expose `start()`, `answer(question)`, and `close()`.
This is a structural `Protocol`; a team is not required to inherit from
a common base class or import DeepEval. The notebook constructs every
wrapper with `importlib.import_module()`.

`answer()` can return a string or a dictionary containing `answer`,
optional `retrieval_context` and optional `tools_called` records.
The reference answer is never supplied to a wrapper.
`tools_called=None` means unreported; `[]` means no tools were called.

Tool records additionally support an optional `arguments_truncated` boolean,
indicating that the SDK deliberately omitted an oversized parameter object.

To customize cli-agent, edit the adjacent `cli_agent_wrapper.py`
implementation or supply constructor arguments through `kwargs`.
`workspace="."` is its default and is **not** the location of the
Golden dataset. Local sibling modules can be imported by their filename
without `.py`, while installed package modules use standard imports.
That module uses `agent.reset()` for conversation independence and
optionally captures selected Knowledge passages and redacted tool arguments.
Every other team's wrapper follows the same lifecycle and response contract.

In [ ]:
class AgentWrapper(Protocol):
    """Small lifecycle expected from all benchmark participants.

    answer() must isolate each Golden question from previous conversations.
    """
    async def start(self) -> None: ...
    async def answer(self, question: str) -> str | Mapping: ...
    async def close(self) -> None: ...


def create_wrapper(spec) -> AgentWrapper:
    """Construct any team's installed wrapper class, without SDK knowledge."""
    entrypoint = spec["entrypoint"]
    if entrypoint.count(":") != 1:
        raise ValueError("entrypoint must be 'python.module:WrapperClass'")
    module_name, class_name = entrypoint.split(":", 1)
    if not module_name or not all(part.isidentifier() for part in module_name.split(".")):
        raise ValueError(f"Invalid wrapper module: {entrypoint!r}")
    if not class_name.isidentifier():
        raise ValueError(f"Invalid wrapper class: {entrypoint!r}")
    # A single-name module beside the notebook can be loaded directly even
    # when the kernel was launched at the repository root. Installed team
    # modules (including dotted package paths) use regular Python imports.
    sibling = NOTEBOOK_DIR / f"{module_name}.py" if "." not in module_name else None
    if sibling is not None and sibling.is_file():
        # A stable, private name avoids colliding with installed packages.
        # Dataclasses and other decorators resolve cls.__module__ via
        # sys.modules during import, so registration must precede execution.
        sibling_name = (
            f"_benchmark_sibling_{module_name}_"
            f"{hashlib.sha256(str(sibling.resolve()).encode('utf-8')).hexdigest()[:16]}"
        )
        module = sys.modules.get(sibling_name)
        if module is None:
            module_spec = importlib.util.spec_from_file_location(sibling_name, sibling)
            if module_spec is None or module_spec.loader is None:
                raise ImportError(f"Unable to load benchmark wrapper: {sibling}")
            module = importlib.util.module_from_spec(module_spec)
            sys.modules[sibling_name] = module
            try:
                module_spec.loader.exec_module(module)
            except BaseException:
                # Do not leave an incompletely imported module behind.
                sys.modules.pop(sibling_name, None)
                raise
    else:
        module = importlib.import_module(module_name)
    wrapper_class = getattr(module, class_name)
    if not inspect.isclass(wrapper_class):
        raise TypeError(f"{entrypoint}: entrypoint must refer to a wrapper class")
    wrapper = wrapper_class(**spec.get("kwargs", {}))
    for method in ("start", "answer", "close"):
        if not callable(getattr(wrapper, method, None)):
            raise TypeError(f"{entrypoint}: missing {method}()")
    return wrapper


def normalize_tool_calls(value):
    """None means no telemetry; [] means telemetry confirms zero calls.

    input_parameters is optional and may be sensitive. Some wrappers
    explicitly opt into diagnostic capture; data is cached privately.
    """
    if value is None:
        return None
    if not isinstance(value, (list, tuple)):
        raise ValueError("tools_called must be a list or None")
    normalized = []
    for tool in value:
        if not isinstance(tool, Mapping):
            raise ValueError("each tools_called entry must be a mapping")
        name = tool.get("name")
        if not isinstance(name, str) or not name.strip():
            raise ValueError("tools_called entries require a nonempty name")
        entry = {"name": name}
        if "input_parameters" in tool:
            params = tool["input_parameters"]
            if params is not None and not isinstance(params, dict):
                raise ValueError("input_parameters must be a dictionary or None")
            if params is not None:
                entry["input_parameters"] = params
        for key in ("call_id", "status"):
            if key in tool and tool[key] is not None:
                if not isinstance(tool[key], str):
                    raise ValueError(f"tools_called.{key} must be a string")
                entry[key] = tool[key]
        if "arguments_truncated" in tool:
            if not isinstance(tool["arguments_truncated"], bool):
                raise ValueError("tools_called.arguments_truncated must be a bool")
            entry["arguments_truncated"] = tool["arguments_truncated"]
        if "duration_ms" in tool and tool["duration_ms"] is not None:
            duration = tool["duration_ms"]
            if (isinstance(duration, bool) or not isinstance(duration, (int, float))
                    or not 0 <= duration < float("inf")):
                raise ValueError("tools_called.duration_ms must be finite and nonnegative")
            entry["duration_ms"] = duration
        # Validate data before it enters our JSON cache; never stringify
        # unknown objects or silently drop argument values.
        try:
            json.dumps(entry, allow_nan=False)
        except (TypeError, ValueError) as exc:
            raise ValueError("tools_called must be JSON-serializable") from exc
        normalized.append(entry)
    return normalized


def normalize_answer(value):
    """Accepted: str or {answer, retrieval_context?, tools_called?}."""
    if isinstance(value, str):
        answer, context, tools = value, None, None
    elif isinstance(value, Mapping):
        answer = value.get("answer")
        context = value.get("retrieval_context")
        tools = normalize_tool_calls(value.get("tools_called"))
    else:
        raise TypeError("Agent must return a string or an answer mapping")
    if not isinstance(answer, str) or not answer.strip():
        raise ValueError("Agent returned an empty answer")
    if context is not None:
        if (not isinstance(context, (list, tuple))
                or any(
                    not isinstance(part, str) or not part.strip()
                    for part in context
                )):
            raise ValueError("retrieval_context must be a list of nonempty strings")
        # No passages is a valid retrieval outcome, not an agent failure.
        context = list(context) or None
    return {
        "actual_output": answer,
        "retrieval_context": context,
        "tools_called": tools,
    }

async def invoke_agent(wrapper, question):
    """Timeout and normalize one independent Golden question."""
    result = wrapper.answer(question)
    if not inspect.isawaitable(result):
        raise TypeError("AgentWrapper.answer must be async")
    return normalize_answer(await asyncio.wait_for(result, timeout=TIMEOUT_SECONDS))

## 4 · Lifecycle and per-agent response caching

The notebook loads every wrapper using the same generic factory, calls
`start()` at the first non-cached Golden, runs `answer(question)` for each
question and calls `close()` in a `finally` block. A failed or timed-out
question causes the wrapper to be discarded and recreated.

All implementations, including `cli_agent_wrapper.py`, must ensure
question isolation and may retain expensive model, connection or index
resources between questions.

Answer caching is independent of DeepEval judge caching. Each agent's
ID, module class, constructor arguments, `revision` and `CACHE_EPOCH`
participate in its response key. Optional `cache_files` hashes arbitrary
local config, system prompt, or other files for **any** agent. This replaces
the former special-case SDK config hashing.

In [ ]:
def agent_fingerprint(name, spec):
    """Generic identity for any team's wrapper; no agent-specific imports."""
    digest = hashlib.sha256(json.dumps({
        "name": name, "entrypoint": spec["entrypoint"],
        "revision": spec["revision"],
        "kwargs": spec.get("kwargs", {}),
        "cache_files": spec.get("cache_files", []),
        "response_schema_version": 4,
        "goldens_file": str(GOLDENS_FILE),
        # A relative agent workspace such as "." resolves from the kernel
        # cwd, which can differ even with the same notebook and Goldens.
        "kernel_cwd": str(Path.cwd().resolve()),
        "epoch": CACHE_EPOCH,
    }, sort_keys=True, allow_nan=False).encode("utf-8"))
    for filename in sorted(spec.get("cache_files", [])):
        filepath = Path(filename).expanduser()
        if filepath.is_symlink():
            raise ValueError(f"Refusing symlink cache file: {filename}")
        filepath = filepath.resolve(strict=True)
        if not filepath.is_file() or filepath.stat().st_size > 1024 * 1024:
            raise ValueError(f"cache_files must reference regular files <= 1 MiB: {filename}")
        digest.update(str(filepath).encode("utf-8"))
        digest.update(filepath.read_bytes())
    return digest.hexdigest()

def response_file(agent_id, spec, golden):
    key = hashlib.sha256(json.dumps([
        agent_fingerprint(agent_id, spec),
        golden.additional_metadata["golden_id"],
        golden.input, golden.expected_output,
    ], ensure_ascii=False).encode("utf-8")).hexdigest()
    return OUTPUT_DIR / "responses" / f"{key}.json"


def read_cached_response(path):
    if not path.is_file() or path.is_symlink() or path.stat().st_size > 16_000_000:
        return None
    try:
        record = json.loads(path.read_text(encoding="utf-8"))
        if isinstance(record, dict):
            return normalize_answer({
                "answer": record.get("actual_output"),
                "retrieval_context": record.get("retrieval_context"),
                "tools_called": record.get("tools_called"),
            })
    except (ValueError, TypeError, OSError):
        pass
    return None


def save_response(path, response):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = None
    try:
        with tempfile.NamedTemporaryFile(
            mode="w", encoding="utf-8", dir=path.parent,
            prefix=".response-", suffix=".tmp", delete=False,
        ) as stream:
            temporary = Path(stream.name)
            os.chmod(temporary, 0o600)
            json.dump(response, stream, ensure_ascii=False)
            stream.flush()
            os.fsync(stream.fileno())
        os.replace(temporary, path)
    finally:
        if temporary is not None:
            temporary.unlink(missing_ok=True)


async def close_wrapper(wrapper):
    """Best-effort cleanup; never conceal an answer exception."""
    if wrapper is None:
        return
    try:
        result = wrapper.close()
        if not inspect.isawaitable(result):
            raise TypeError("AgentWrapper.close must be async")
        await asyncio.wait_for(result, timeout=TIMEOUT_SECONDS)
    except Exception as error:
        print(f"Wrapper cleanup failed: {type(error).__name__}")


all_responses = {}
all_failures = {}
for agent_id, spec in AGENTS.items():
    responses, failures = {}, {}
    wrapper = None
    try:
        for golden in dataset.goldens:
            golden_id = golden.additional_metadata["golden_id"]
            checkpoint = response_file(agent_id, spec, golden)
            cached = None if FRESH_RESPONSES else read_cached_response(checkpoint)
            if cached is not None:
                responses[golden_id] = cached
                continue
            try:
                if wrapper is None:
                    wrapper = create_wrapper(spec)
                    started = wrapper.start()
                    if not inspect.isawaitable(started):
                        raise TypeError("AgentWrapper.start must be async")
                    await asyncio.wait_for(started, timeout=TIMEOUT_SECONDS)
                response = await invoke_agent(wrapper, golden.input)
                save_response(checkpoint, response)
                responses[golden_id] = response
            except Exception as error:
                # A failed request may leave model/tool state unusable.
                # Discard the wrapper and create a fresh one for next case.
                failures[golden_id] = type(error).__name__
                await close_wrapper(wrapper)
                wrapper = None
    finally:
        await close_wrapper(wrapper)
        all_responses[agent_id] = responses
        all_failures[agent_id] = failures
    print(f"{agent_id}: {len(responses)}/{len(dataset.goldens)} answered; "
          f"{len(failures)} failed")
    if failures:
        print("  Failed Golden IDs:", ", ".join(sorted(failures)))

## 5 · DeepEval judges answers, optionally retrieval and tool correctness

Every team gets the **same reference answers**, Judge model and shared
factual metrics. Retrieval metrics use only passages the wrapper actually
provides. Tool metrics use actual `tools_called` only when a Golden
**explicitly** specifies `expected_tools`. Our generated Goldens do
not currently specify expected tools, so tool correctness is **not**
automatically run. This avoids ranking heterogeneous agents by a fixed
tool implementation.

DeepEval's `ToolCorrectnessMetric` compares tool names by default;
`TOOL_COMPARE_ARGUMENTS=True` enables comparison of input parameters
when both actual and expected tool arguments are supplied. Tool counts,
statuses and durations are reported independently of Golden expectations.
`ToolUseMetric` instead targets multi-turn conversations and is not run
on these single-question Goldens.

In [ ]:
from deepeval import evaluate
from deepeval.metrics import (
    GEval, AnswerRelevancyMetric, FaithfulnessMetric,
    ContextualRecallMetric, ContextualPrecisionMetric, ToolCorrectnessMetric,
)
from deepeval.test_case import LLMTestCase, SingleTurnParams, ToolCall, ToolCallParams
from deepeval.evaluate import AsyncConfig, CacheConfig, DisplayConfig

evaluation_options = {
    "cache_config": CacheConfig(use_cache=not FRESH_EVALUATION, write_cache=True),
    "display_config": DisplayConfig(
        results_folder=str(OUTPUT_DIR / "deepeval"),
        inspect_after_run=False,
    ),
}

evaluation_results = {}
for agent_id, spec in AGENTS.items():
    responses = all_responses[agent_id]
    answer_cases, retrieval_cases, tool_cases = [], [], []
    for golden in dataset.goldens:
        response = responses.get(golden.additional_metadata["golden_id"])
        if response is None:
            continue  # failures are counted separately in the summary
        base = {
            "input": golden.input,
            "expected_output": golden.expected_output,
            "actual_output": response["actual_output"],
        }
        answer_cases.append(LLMTestCase(**base))
        if response["retrieval_context"]:
            retrieval_cases.append(LLMTestCase(
                **base, retrieval_context=response["retrieval_context"],
            ))
        # Tool correctness needs a known expected tool set. Never infer one
        # from the answer or from another team's implementation.
        if response["tools_called"] is not None and golden.expected_tools is not None:
            tool_cases.append(LLMTestCase(
                **base,
                tools_called=[
                    ToolCall(
                        name=call["name"],
                        **({"input_parameters": call["input_parameters"]}
                           if "input_parameters" in call else {}),
                    ) for call in response["tools_called"]
                ],
                expected_tools=golden.expected_tools,
            ))

    correctness = GEval(
        name="Answer correctness",
        criteria=(
            "Judge factual correctness and completeness against the fixed reference. "
            "Accept paraphrases, short or detailed answers equally when all "
            "necessary facts and caveats are preserved. Penalize invented facts."
        ),
        evaluation_params=[
            SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT,
            SingleTurnParams.EXPECTED_OUTPUT,
        ],
        model=judge, threshold=THRESHOLD,
    )
    metrics = [
        correctness,
        AnswerRelevancyMetric(model=judge, threshold=THRESHOLD),
    ]
    if spec.get("style_criteria"):
        metrics.append(GEval(
            name="Style compliance",
            criteria=(
                "Score only response style, not factual correctness. Goal: "
                + spec["style_criteria"]
            ),
            evaluation_params=[
                SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT,
            ],
            model=judge, threshold=THRESHOLD,
        ))

    answer_results = evaluate(
        test_cases=answer_cases, metrics=metrics,
        identifier=f"rag-{agent_id}-answer",
        async_config=AsyncConfig(max_concurrent=JUDGE_MAX_CONCURRENT),
        **evaluation_options,
    ) if answer_cases else None

    retrieval_results = None
    if retrieval_cases:
        uncertainty = GEval(
            name="Uncertainty calibration",
            criteria=(
                "Using the actual retrieved passages, assess justified "
                "confidence or limitations in the answer. Penalize unsupported "
                "claims and unnecessary refusal when retrieved evidence supports "
                "an answer. The reference is the factual target, NOT proof that "
                "the agent retrieved it."
            ),
            evaluation_params=[
                SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT,
                SingleTurnParams.EXPECTED_OUTPUT, SingleTurnParams.RETRIEVAL_CONTEXT,
            ],
            model=judge, threshold=THRESHOLD,
        )
        retrieval_results = evaluate(
            test_cases=retrieval_cases,
            metrics=[
                FaithfulnessMetric(model=judge, threshold=THRESHOLD),
                ContextualRecallMetric(model=judge, threshold=THRESHOLD),
                ContextualPrecisionMetric(model=judge, threshold=THRESHOLD),
                uncertainty,
            ],
            identifier=f"rag-{agent_id}-retrieval",
            async_config=AsyncConfig(max_concurrent=JUDGE_MAX_CONCURRENT),
            **evaluation_options,
        )
    tool_results = None
    if tool_cases:
        tool_results = evaluate(
            test_cases=tool_cases,
            metrics=[
                ToolCorrectnessMetric(
                    model=judge, threshold=THRESHOLD,
                    evaluation_params=(
                        [ToolCallParams.INPUT_PARAMETERS]
                        if TOOL_COMPARE_ARGUMENTS else []
                    ),
                ),
            ],
            identifier=f"rag-{agent_id}-tools",
            async_config=AsyncConfig(max_concurrent=JUDGE_MAX_CONCURRENT),
            **evaluation_options,
        )
    evaluation_results[agent_id] = {
        "answer": answer_results,
        "retrieval": retrieval_results,
        "tools": tool_results,
        "successful_cases": len(answer_cases),
        "retrieval_cases": len(retrieval_cases),
        "tool_cases": len(tool_cases),
    }

## 6 · Compare scores, tool activity and coverage

The comparison table shows all agent failures and the number of Goldens
actually answered, so that incomplete runs cannot appear superior merely
because they skipped hard questions.

Tool metadata is **optional**: `tool_telemetry_cases` distinguishes teams
that report actual tool usage from teams that do not; `total_tool_calls`,
error statuses and argument coverage only describe reported records.
Tool correctness scores exist only for Goldens explicitly defining
expected tools and for agents reporting tool telemetry. Technical tool names
may differ across agent implementations; compare these scores only where
the Golden's tool expectations make sense for every participant.

In [ ]:
from collections import defaultdict
import pandas as pd
from IPython.display import display

summary_rows = []
for agent_id, results in evaluation_results.items():
    scores = defaultdict(list)
    judge_errors = defaultdict(int)
    for section in ("answer", "retrieval", "tools"):
        run = results[section]
        for testcase in (getattr(run, "test_results", []) if run else []):
            for metric in testcase.metrics_data or []:
                name = metric.name.removesuffix(" [GEval]")
                if metric.error or metric.score is None:
                    judge_errors[name] += 1
                else:
                    scores[name].append(float(metric.score))
    row = {
        "agent": agent_id,
        "goldens": len(dataset.goldens),
        "answered": results["successful_cases"],
        "agent_failures": len(all_failures[agent_id]),
        "retrieval_cases": results["retrieval_cases"],
        "tool_correctness_cases": results["tool_cases"],
        "tool_telemetry_cases": sum(
            response["tools_called"] is not None
            for response in all_responses[agent_id].values()
        ),
        "total_tool_calls": sum(
            len(response["tools_called"] or [])
            for response in all_responses[agent_id].values()
        ),
        "tool_error_calls": sum(
            call.get("status") in {"error", "failed", "interrupted"}
            for response in all_responses[agent_id].values()
            for call in (response["tools_called"] or [])
        ),
        "tool_calls_with_arguments": sum(
            "input_parameters" in call
            for response in all_responses[agent_id].values()
            for call in (response["tools_called"] or [])
        ),
    }
    for name in (
        "Answer correctness", "Answer Relevancy",
        "Faithfulness", "Contextual Recall", "Contextual Precision",
        "Uncertainty calibration", "Style compliance",
        "Tool Correctness",
    ):
        values = scores.get(name, [])
        row[name] = round(sum(values) / len(values), 3) if values else None
        row[name + " judged"] = len(values)
        row[name + " judge_errors"] = judge_errors[name]
    summary_rows.append(row)

display(pd.DataFrame(summary_rows))
print("Reports:", OUTPUT_DIR / "deepeval")

### Interpretation and reproducibility

- Everyone uses the same Golden questions, unchanged references, judge
  model and shared factual rubrics. Every wrapper isolates question histories
  and is started once per uninterrupted set of uncached questions.
- For controlled comparisons, use a fixed corpus/index snapshot, model
  settings and adapter `revision`, and set `MAX_CASES=None`.
- A result without `retrieval_context` can still be judged for correctness,
  but not for actual retrieval grounding. Never substitute Golden quotations.
- Tool telemetry is optional. The sibling `cli_agent_wrapper.py` shows
  an explicit SDK tool-argument trace opt-in, with bounded, best-effort
  redacted inputs. Tool correctness requires explicit `expected_tools` in
  the Golden; our generated dataset does not currently define them.
- No SDK imports or special branches are needed in this notebook. All
  wrappers are imported by the same factory, from the notebook's sibling
  directory or installed Python packages. `cache_files` works for all
  agents; a modified configuration must invalidate response checkpoints.
- The benchmark resolves `GOLDENS_FILE` independently of an agent's
  `workspace`. Override the dataset path directly when exporting elsewhere.
- Changing another team's implementation, prompt, model or external index
  requires an updated `revision` / `CACHE_EPOCH` or
  `FRESH_RESPONSES=True`. Judge cache is independent.
- Stored responses and DeepEval reports may contain sensitive data. Keep
  `rag-goldens/evaluation/` out of version control.